In [ ]:
# ==============================
# 🔹 STEP 1: INSTALL LIBRARIES
# ==============================
!pip install -U pandas matplotlib seaborn slack-sdk kagglehub langchain-openai


# ==============================
# 🔹 STEP 2: SET API KEYS
# ==============================
import os

os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_KEY"
os.environ["SLACK_USER_TOKEN"] = "YOUR_SLACK_TOKEN"

CHANNEL_ID = "YOUR_CHANNEL_ID"

print("✅ Keys set")


# ==============================
# 🔹 STEP 3: SLACK FUNCTION
# ==============================
from slack_sdk import WebClient

slack_client = WebClient(token=os.environ["SLACK_USER_TOKEN"])

def send_to_slack(text, file_path=None):
    try:
        if file_path:
            with open(file_path, "rb") as f:
                slack_client.files_upload_v2(
                    channel=CHANNEL_ID,
                    file=f,
                    initial_comment=text,
                )
        else:
            slack_client.chat_postMessage(channel=CHANNEL_ID, text=text)

        print("✅ Sent to Slack")

    except Exception as e:
        print("❌ Slack Error:", e)


# ==============================
# 🔹 STEP 4: DOWNLOAD DATASET
# ==============================
import kagglehub
import os

path = kagglehub.dataset_download("csafrit2/steel-industry-energy-consumption")

csv_file = None
for root, _, files in os.walk(path):
    for file in files:
        if file.endswith(".csv"):
            csv_file = os.path.join(root, file)

print("📂 Dataset:", csv_file)


# ==============================
# 🔹 STEP 5: LOAD DATA
# ==============================
import pandas as pd

df = pd.read_csv(csv_file)

print("📊 Dataset Overview")
display(df.head())

print("\n📊 Data Info")
df.info()

print("\n📊 Summary")
display(df.describe())


# ==============================
# 🔹 STEP 6: VISUALIZATIONS
# ==============================
import matplotlib.pyplot as plt
import seaborn as sns

# Trend
plt.figure()
df['Usage_kWh'].plot()
plt.title("Energy Consumption Trend")
plt.savefig("trend.png")

# Load Type
plt.figure()
df['Load_Type'].value_counts().plot(kind='bar')
plt.title("Load Type Distribution")
plt.savefig("load_type.png")

# Correlation
plt.figure()
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.savefig("correlation.png")

# Peak Usage
top_usage = df.nlargest(10, 'Usage_kWh')

plt.figure()
top_usage['Usage_kWh'].plot(kind='bar')
plt.title("Top Energy Usage")
plt.savefig("top_usage.png")

# Distribution
plt.figure()
sns.histplot(df['Usage_kWh'], bins=30)
plt.title("Energy Distribution")
plt.savefig("distribution.png")

print("✅ Charts Created")


# ==============================
# 🔹 STEP 7: AI INSIGHTS (LANGCHAIN)
# ==============================
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

prompt = f"""
You are a senior data analyst working in a steel manufacturing company.

Analyze ONLY the dataset below.

Summary:
{df.describe().to_string()}

Top 5 Usage:
{df.nlargest(5, 'Usage_kWh').to_string()}

Instructions:
- Use actual numbers
- No generic theory
- Identify patterns & anomalies
- Give business insights

Output:
1. Key Insights
2. Anomaly Analysis
3. Recommendations
"""

response = llm.invoke(prompt)
ai_insights = response.content

print(ai_insights)


# ==============================
# 🔹 STEP 8: SEND TO SLACK
# ==============================
send_to_slack("📊 Steel Plant Energy Analysis Report")
send_to_slack(ai_insights)

send_to_slack("📈 Trend", "trend.png")
send_to_slack("📊 Load Type", "load_type.png")
send_to_slack("🔍 Correlation", "correlation.png")
send_to_slack("⚠️ Peak Usage", "top_usage.png")
send_to_slack("📉 Distribution", "distribution.png")

print("🚀 DONE!")